<a href="https://colab.research.google.com/github/toothlesscorn/MonoGame/blob/develop/notebooks/Quick_Primer_on_Colab_Jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -qq -y install fonts-nanum > /dev/null

In [ ]:
# 1. 라이브러리 설치 및 임포트
!pip install koreanize-matplotlib
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

# 2. 구글 계정 인증 및 시트 로드
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

sheet_url = 'https://docs.google.com/spreadsheets/d/1NDVinQpHBUNopCtt03p8ObE3wWkFQr31i3-LhtBR2eI/edit?gid=1412239890#gid=1412239890'
doc = gc.open_by_url(sheet_url)
sheet = doc.get_worksheet(1)  # 두 번째 탭
all_values = sheet.get_all_values()

# 3. 'Date' 헤더 위치 자동 찾기 (성공했던 로직)
header_idx = 0
for i, row in enumerate(all_values):
    if 'Date' in row:
        header_idx = i
        break

# 4. 데이터프레임 생성 및 정제
df = pd.DataFrame(all_values[header_idx+1:], columns=all_values[header_idx])
df.columns = df.columns.str.strip()  # 컬럼명 공백 제거
df = df.replace('', '0')           # 빈칸 0 처리

# 5. 숫자 데이터 변환 (ROI 및 비용 지표)
# ROI 증분 지표와 비용, 매출 지표를 모두 숫자로 변환합니다.
roas_days_cols = [
    'Net Total Revenue ROI 1d', 'Net Total Revenue ROI 7d',
    'Net Total Revenue ROI 14d', 'Net Total Revenue ROI 30d',
    'Net Total Revenue ROI 60d', 'Net Total Revenue ROI 90d',
    'Net Total Revenue ROI 180d'
]
basic_cols = ['Cost', 'Net Total Revenue ROI ACTUAL'] # 실제 시트 컬럼명에 맞춤

for col in (roas_days_cols + basic_cols):
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '').str.replace('%', '')
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# ---------------------------------------------------------
# 6. 시각화 시작
# ---------------------------------------------------------

# 차트 1: 광고비(Cost) vs 실질 매출 ROI
fig, ax1 = plt.subplots(figsize=(12, 6))

# 비용 (막대)
ax1.bar(df['Date'], df['Cost'], color='lightgrey', label='광고비 (Cost)', alpha=0.7)
ax1.set_ylabel('Cost Amount')
ax1.set_xlabel('Date Range')

# 실질 매출 ROI (선) - 실제 수익률 확인
ax2 = ax1.twinx()
ax2.plot(df['Date'], df['Net Total Revenue ROI ACTUAL'], color='blue', marker='o', label='Actual ROI')
ax2.set_ylabel('Actual ROI (Cumulative)')
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='BEP(100%)')

plt.title('주차별 광고비 대비 실질 ROI 추이', fontsize=15)
plt.xticks(rotation=45)
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

# 차트 2: ROAS 증분 (D1 -> D180) - 전체 데이터의 평균적 성장 곡선
plt.figure(figsize=(10, 5))

# 실제 존재하는 ROI 컬럼만 추출하여 평균 계산
existing_roas_cols = [c for c in roas_days_cols if c in df.columns]
roas_values = df[existing_roas_cols].mean()
short_labels = ['D1', 'D7', 'D14', 'D30', 'D60', 'D90', 'D180'][:len(roas_values)]

plt.plot(short_labels, roas_values, marker='s', color='green', linestyle='-', linewidth=2)
plt.axhline(y=1.0, color='red', linestyle='--', label='BEP (100%)')

plt.title('D1 -> D180 ROAS 누적 증분 성과 (평균)', fontsize=15)
plt.ylabel('누적 성과 (Ratio)')
plt.grid(True, axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
!pip install koreanize-matplotlib
import koreanize_matplotlib

# 1. 데이터 가져오기 (전체 값을 리스트로 먼저 받음)
all_values = sheet.get_all_values()

# 2. 'Date'라는 단어가 포함된 행(Header) 찾기
header_idx = 0
for i, row in enumerate(all_values):
    if 'Date' in row:
        header_idx = i
        break

# 3. 찾은 행을 기준으로 데이터프레임 생성
df = pd.DataFrame(all_values[header_idx+1:], columns=all_values[header_idx])

# 4. 공백 제거 및 데이터 정제
df.columns = df.columns.str.strip() # 컬럼명 공백 제거
df = df.replace('', '0') # 빈칸은 0으로 채움

# 5. 숫자 데이터 변환 (Cost, Revenue 등)
# 이미지에 보이는 컬럼들 중 숫자로 쓸 것들만 변환
numeric_cols = ['Cost', 'Net Total Revenue ROI ACTUAL', 'Installs'] # 더 필요한 컬럼 추가 가능
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].str.replace(',', '').str.replace('%', '')
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# ---------------------------------------------------------
# 6. 시각화 (날짜 범위 문자열 그대로 사용)
# ---------------------------------------------------------
plt.figure(figsize=(15, 7))

# X축은 '2026-01-19 ... 2026-01-24' 이 문자열을 그대로 사용합니다.
plt.bar(df['Date'], df['Cost'], color='lightgrey', label='광고비 (Cost)')

# 가독성을 위해 최신 데이터가 오른쪽으로 오게 정렬 (이미지가 역순이라면 필요)
# df = df.iloc[::-1] # 데이터가 과거순으로 되어있다면 주석 해제

plt.title('주차별 광고비 집행 현황', fontsize=16)
plt.xlabel('주간 범위 (Date Range)')
plt.ylabel('Cost')
plt.xticks(rotation=45) # 글자가 기니까 대각선으로 돌려줍니다.
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. 라이브러리 임포트 및 구글 시트 인증
from google.colab import auth
import gspread
from google.auth import compute_engine
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 구글 계정 인증
auth.authenticate_user()
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# 2. 구글 시트 데이터 불러오기
# '시트_이름_또는_URL' 부분에 실제 파일 이름을 넣으세요.
sheet_url = 'https://docs.google.com/spreadsheets/d/1NDVinQpHBUNopCtt03p8ObE3wWkFQr31i3-LhtBR2eI/edit?gid=1412239890#gid=1412239890'
doc = gc.open_by_url(sheet_url)
sheet = doc.get_worksheet(1)
data = sheet.get_all_values()
df = pd.DataFrame(data)

# 데이터 확인 (컬럼명이 맞는지 확인용)
print(df.head())

# ---------------------------------------------------------
# 3. 시각화 (데이터 구조에 따라 컬럼명 수정 필요)
# ---------------------------------------------------------

# 그래프 한글 깨짐 방지 (필요 시)
plt.rc('font', family='NanumBarunGothic')

# 차트 1: Cost vs Net Total Revenue Actual (막대+선 혼합)
fig, ax1 = plt.subplots(figsize=(12, 6))

# 비용 (Cost) - 막대 그래프
ax1.bar(df['Date'], df['Cost'], color='lightgrey', label='Cost', alpha=0.7)
ax1.set_xlabel('Date')
ax1.set_ylabel('Cost / Revenue Amount', color='black')

# 실질 총 매출 (Net Total Revenue Actual) - 선 그래프
ax1.plot(df['Date'], df['Net Total Revenue Actual'], color='blue', marker='o', label='Net Revenue', linewidth=2)

# 차트 2: ROAS 증분 (D1 -> D180)
# 데이터 구조가 'D1', 'D7', 'D30', 'D180' 등의 컬럼으로 되어 있다고 가정
roas_days = ['D1', 'D7', 'D14', 'D30', 'D60', 'D90', 'D180']
# 가장 최근 날짜의 코호트 데이터를 보거나, 평균치를 봅니다.
roas_values = df[roas_days].mean() # 전체 평균 ROAS 증분

plt.figure(figsize=(10, 5))
plt.plot(roas_days, roas_values, marker='s', color='green', linestyle='-', linewidth=2)
plt.axhline(y=1.0, color='red', linestyle='--', label='BEP (100%)') # 손익분기선 (비율일 경우 1.0, %일 경우 100)
plt.title('ROAS Growth Trend (D1 to D180)')
plt.ylabel('Cumulative ROAS')
plt.grid(True, axis='y', alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()